In [1]:
%load_ext autoreload
%autoreload 2

import yaml
import logging 
import sys
from agentz.pydantic_models import ExperimentConfiguration, ExecutedChunk
from agentz.utility import print_dict
from agentz import Agent
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt

import time
from agentz.pydantic_models import Episode


logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setLevel(logging.DEBUG)
if not any(isinstance(h, logging.StreamHandler) for h in logger.handlers):
    logger.addHandler(stream_handler)
    
def load_settings(path : str) -> dict:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}")
    with path.open("r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}

    print_dict(cfg)
    return ExperimentConfiguration(**cfg)  

task = {
    "id": "94d95f96-9699-4208-98ba-3c3119edf9c2",
    "instruction": "I want to install Spotify on my current system. Could you please help me?",
    "config": [
        {
            "type": "execute",
            "parameters": {
                "command": [
                    "python",
                    "-c",
                    "import pyautogui; import time; pyautogui.click(960, 540); time.sleep(0.5);"
                ]
            }
        }
    ],
    "evaluator": {
        "func": "check_include_exclude",
        "result": {
            "type": "vm_command_line",
            "command": "which spotify"
        },
        "expected": {
            "type": "rule",
            "rules": {
                "include": ["spotify"],
                "exclude": ["not found"]
            }
        }
    }
}

SETTINGS_FILE = "../agent/config_files/start_agent_nb.yaml"
settings = load_settings(SETTINGS_FILE)
settings = load_settings(SETTINGS_FILE)
agent = await Agent.create("my-agent", settings)
perception = agent.env.reset(task)
env = agent.env  # OSWorldEnvironment



- log_dir: ../data/logs/agent
- log_level: INFO
- mem_interval_sec: 1
- gpt_client_settings:
    - endpoint: https://rt-bdi-zanolo.openai.azure.com/
    - api_version: 2024-12-01-preview
    - model: gpt-4o
    - max_tokens: 1000
    - temperature: 0.0
    - max_retries: 2
    - reasoning:
        - effort: low
    - delay_seconds: 1.0
    - gpt_log_path: ../data/gpt_interactions
- osworld_settings:
    - provider_name: vmware
    - action_space: pyautogui
    - region: EU
    - path_to_vm: C:/Users/Luca/Desktop/thesis/data/vms/Ubuntu-OSWorld/Ubuntu-OSWorld.vmx
    - snapshot_name: init_state
    - cache_dir: cache
    - screen_size:
        • 1920
        • 1080
    - headless: True
    - require_a11y_tree: True
    - require_terminal: True
    - os_type: Ubuntu
    - enable_proxy: False
    - client_username: user
    - client_password: password
- aci_settings:
    - ocr_method: tesseract
- memory_settings:
    - memory_name: mem00
    - root: ../agent/memories
    - initialize_memor

`torch_dtype` is deprecated! Use `dtype` instead!


Initializing OCR...
Memory initialized. mem_root=../agent/memories
Memory enabled (external session factory). mem_root=../agent/memories
Agent initialized. env.ready=True


TimeoutError: timed out

In [ ]:
import textwrap
import pprint
import numpy as np

failed_tests = []

pp = pprint.PrettyPrinter(indent=4, width=100, sort_dicts=False)

def header(title: str):
    print("\n" + "=" * 12 + f" {title} " + "=" * 12)

def mark_test(name: str, ok: bool):
    icon = "✅" if ok else "❌"
    print(f"{icon} {name}")
    if not ok:
        failed_tests.append(name)
        


# =========================
# 4) EVALUATE
# =========================
header("EVALUATE")

try:
    resp = env.evaluate()

    status_ok = resp.get("status") == "ok"
    result = resp.get("result", {})
    metric_present = isinstance(result, dict) and "metric" in result
    metric = result.get("metric")

    metric_ok = isinstance(metric, (int, float, bool, type(None)))
    ok = status_ok and metric_present and metric_ok

    mark_test("evaluate()", ok)

    print("   Raw response:")
    pp.pprint(resp)

except Exception as e:
    mark_test("evaluate()", False)
    print(f"   Exception: {e!r}")
    
# =========================
# 1) VM INFO
# =========================
header("VM INFO")

try:
    vm = env.vm_info()  # dict: {'status','platform','screen_size',...}

    status_ok = vm.get("status") == "ok"
    platform = vm.get("platform")
    screen = vm.get("screen_size", {})

    screen_ok = isinstance(screen, dict) and "width" in screen and "height" in screen
    ok = status_ok and isinstance(platform, str) and screen_ok

    mark_test("vm_info()", ok)

    print("   Raw response:")
    pp.pprint(vm)

except Exception as e:
    mark_test("vm_info()", False)
    print(f"   Exception: {e!r}")


# =========================
# 2) OBSERVE
# =========================
header("OBSERVE")

try:
    obs = env.observe()
    img = obs.get("screenshot")
    a11y = obs.get("accessibility_tree")
    term = obs.get("terminal")

    img_ok = isinstance(img, np.ndarray) and img.ndim == 3 and img.dtype == np.uint8
    a11y_ok = a11y is None or isinstance(a11y, (dict, list, str))
    term_ok = term is None or isinstance(term, (str, dict))

    ok = img_ok and a11y_ok and term_ok

    mark_test("observe()", ok)

    shape = getattr(img, "shape", None)
    print("   screenshot:")
    print(f"      type={type(img)}, shape={shape}, dtype={getattr(img,'dtype',None)}")
    print("   accessibility_tree:", "present" if a11y is not None else "None")
    print("   terminal:", type(term).__name__ if term is not None else "None")

except Exception as e:
    mark_test("observe()", False)
    print(f"   Exception: {e!r}")


# =========================
# 3) PYTHON EXEC
#   (usa solo il parametro 'code')
# =========================
header("PYTHON EXEC")

try:
    code = textwrap.dedent("""
        import platform
        print("hello from python_exec")
        print(platform.system())
    """).strip()

    resp = env.python_exec(code)

    outer_status = resp.get("status")
    result = resp.get("result", {})
    inner_status = result.get("status")
    rc = result.get("returncode")

    ok = (outer_status == "ok" and inner_status == "success" and rc == 0)

    mark_test("python_exec()", ok)

    print("   Raw response:")
    pp.pprint(resp)

except Exception as e:
    mark_test("python_exec()", False)
    print(f"   Exception: {e!r}")


# =========================
# 4) PYTHON SCRIPT
#   (usa solo lo script come stringa)
# =========================
header("PYTHON SCRIPT")

try:
    remote_path = "/tmp/agentz_osworld_test.txt"
    code = textwrap.dedent(f"""
        import platform
        msg = "hello from python_script!\\n"
        with open("{remote_path}", "w", encoding="utf-8") as f:
            f.write(msg)
        print("WROTE: {remote_path} ON:", platform.system())
    """).strip()

    resp = env.python_script(code)

    outer_status = resp.get("status")
    result = resp.get("result", {})
    inner_status = result.get("status")
    rc = result.get("return_code")  # come nel log precedente

    ok = (outer_status == "ok" and inner_status == "success" and rc == 0)

    mark_test("python_script()", ok)

    print("   Raw response:")
    pp.pprint(resp)

except Exception as e:
    mark_test("python_script()", False)
    print(f"   Exception: {e!r}")


# =========================
# 5) GET FILE
# =========================
header("GET FILE")

try:
    remote_path = "/tmp/agentz_osworld_test.txt"
    data = env.get_file(remote_path, decode=True)  # bytes or None

    ok = isinstance(data, (bytes, bytearray)) and len(data) > 0

    mark_test("get_file()", ok)

    print(f"   bytes: {len(data) if data is not None else 'None'}")
    if isinstance(data, (bytes, bytearray)):
        preview = data[:80].decode("utf-8", errors="replace")
        print("   preview:")
        print(f"      {preview!r}")

except Exception as e:
    mark_test("get_file()", False)
    print(f"   Exception: {e!r}")


# =========================
# 6) BASH SCRIPT
# =========================
header("BASH SCRIPT")

try:
    bash_code = textwrap.dedent("""
        echo "hello from bash_script"
        uname -a
    """).strip()

    resp = env.bash_script(bash_code, timeout=15)

    outer_status = resp.get("status")
    result = resp.get("result", {})
    inner_status = result.get("status")
    rc = result.get("returncode")

    ok = (outer_status == "ok" and inner_status == "success" and rc == 0)

    mark_test("bash_script()", ok)

    print("   Raw response:")
    pp.pprint(resp)

except Exception as e:
    mark_test("bash_script()", False)
    print(f"   Exception: {e!r}")


# =========================
# SUMMARY
# =========================
header("SUMMARY")

if not failed_tests:
    print("🎉 All IPC tests PASSED (status + contenuto).")
else:
    print(f"⚠️ {len(failed_tests)} test(s) reported issues:")
    for name in failed_tests:
        print(f"   - {name}")


In [ ]:
header("OBSERVE")

try:
    obs = env.observe()
    img = obs.get("screenshot")
    a11y = obs.get("accessibility_tree")
    term = obs.get("terminal")

    img_ok = isinstance(img, np.ndarray) and img.ndim == 3 and img.dtype == np.uint8
    a11y_ok = a11y is None or isinstance(a11y, (dict, list, str))
    term_ok = term is None or isinstance(term, (str, dict))

    ok = img_ok and a11y_ok and term_ok

    mark_test("observe()", ok)

    shape = getattr(img, "shape", None)
    print("   screenshot:")
    print(f"      type={type(img)}, shape={shape}, dtype={getattr(img,'dtype',None)}")
    print("   accessibility_tree:", "present" if a11y is not None else "None")
    print("   terminal:", type(term).__name__ if term is not None else "None")
    print(term)
    
except Exception as e:
    mark_test("observe()", False)
    print(f"   Exception: {e!r}")

In [ ]:
Deskt